In [1]:
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import os
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

In [2]:
#fig_name = r'2023年12月结算'
folder = r'D:\TBEA_WHX\TBEA-结算单\TBEA结算单'
integration_path = os.path.join(folder, r'2024年1月-2025年6月结算单汇总.xlsx')
site_company_dict = {
    '振超': '哈密市振超风力发电有限公司', 
    '布尔津': '布尔津县晶能风力发电有限责任公司', 
    '景峡': '哈密风尚发电有限责任公司', 
    '雅满苏': '哈密华风新能源发电有限公司', 
    '十三间房': '哈密十三间房新特风能有限责任公司', 
    '柯坪': '柯坪县柯特新能源有限责任公司', 
    '嘉裕大石头': '木垒县嘉裕风晟发电有限公司', 
    '老君庙': '木垒县新特汇能发电有限责任公司', 
    '莎车': '莎车县新尚能源发电有限责任公司', 
    '中闽大石头': '中闽（木垒）风电有限公司', 
    '若羌': '若羌县卓尚新能源有限公司', 
    #'托里': '乌鲁木齐县君盛风力发电有限公司'
}

In [3]:
site_name = '振超'
month_start = '2025-01'
month_end = '2025-06'

In [4]:
df_primary = pd.read_excel(integration_path, sheet_name=f'{site_name}_主表', 
                            dtype={'结算科目编码': str})
df_attached = pd.read_excel(integration_path, sheet_name=f'{site_name}_附表', 
                            dtype={'结算科目编码': str})
df_primary = df_primary[(df_primary['日期'] >= month_start) & (df_primary['日期'] <= month_end)]
df_attached = df_attached[(df_attached['日期'] >= month_start) & (df_attached['日期'] <= month_end)]

In [5]:
# 计算每种结算科目的电费
def sum_by_subject(group):
    new_group = {'结算科目编码':None, '结算电费':None}
    new_group['结算科目编码'] = group['结算科目编码'].iloc[0]
    new_group['结算电费'] = group['结算电费'].sum()
    new_group[r'结算电量/容量'] = group[r'结算电量/容量'].sum()

    return pd.Series(new_group)

# 从总表和附表中
df_for_plot = df_attached.groupby('结算科目').apply(sum_by_subject).reset_index()
df_primary_tmp = df_primary.groupby('结算科目').apply(sum_by_subject).reset_index()
df_for_plot = pd.concat([df_for_plot, df_primary_tmp], axis=0)
df_for_plot = df_for_plot.sort_values(by='结算科目编码')
df_for_plot = df_for_plot.drop_duplicates(subset=['结算科目', '结算科目编码'], keep='first')
df_for_plot[:3]

,结算科目,结算科目编码,结算电费,结算电量/容量
30,电量清分,01,2119453.35,11914.350
1,中长期交易,0101,2246060.31,11224.473
2,优先发购电量交易,010101,1496030.50,5984.122


In [6]:
# 功能函数：计算节点的结算电费总和
def calculate_node_fee(code):
    if G.nodes[code]['node_fee'] != 0:
        return G.nodes[code]['node_fee']
    else:
        # 计算次一级子节点的电费总和
        # 如04的电费总和 = 0401的电费总和 + 0402的电费总和 + 0403的电费总和
        children_fees = [calculate_node_fee(child) for child in G.successors(code)]
        return sum(children_fees)
    
# 功能函数：计算节点的结算电量总和
def calculate_node_cap(code):
    if G.nodes[code]['node_cap'] != 0:
        return G.nodes[code]['node_cap']
    else:
        # 计算次一级子节点的结算电量总和
        # 如04的电费总和 = 0401的电量总和 + 0402的电量总和 + 0403的电量总和
        children_cap = [calculate_node_cap(child) for child in G.successors(code)]
        return sum(children_cap)
    
# 功能函数：添加节点及其所有父节点
def add_nodes_and_edges(code, name=None, node_fee=0, node_cap=0):
    # 如果是一级节点，则直接连接到 "0"
    if len(code) == 2 and code not in G:
        if name is not None:
            G.add_node(code, label=f"{code}\n{name}", node_fee=node_fee, node_cap=node_cap)
        else:
            G.add_node(code, label=f"{code}", node_fee=node_fee, node_cap=node_cap)
        G.add_edge("0", code)
    # 递归添加所有父节点
    elif len(code) > 2 and code[:-2] not in G:
        add_nodes_and_edges(code[:-2])
    # 添加当前节点
    if code not in G:
        label = f"{code}\n{name}" if name else code
        G.add_node(code, label=label, node_fee=node_fee, node_cap=node_cap)
    # 添加边
    if len(code) > 2:
        G.add_edge(code[:-2], code)

In [7]:
# 将结算单数据整理成表格
def summary_settlement(df_settlement):
    # df_settlement：结算单数据
    # 其中主要内容包括：结算科目、结算科目编码、结算电量/容量、结算电费等列

    # 统计一下各节点的电费、电量、电价等信息，汇集成表格
    # 统计中长期交易(节点0101)开始的三级节点的数据，如保障性发电量(01010101)
    # 统计除中长期交易(节点0101)以外所有的全图二级节点数据，如辅助服务交易(0103)
    df_summary = pd.DataFrame(columns=['结算科目', '结算科目编码', r'电量(万kWh)', r'电价(元/kWh)', '结算电费(万元)'])

    for index, row in df_settlement.iterrows():
        code = row['结算科目编码']
        if (code.startswith('0101') and len(code) == 8) or (len(code) == 4 and code != '0101'):
            # 计算电价
            price = row['结算电费'] / row['结算电量/容量'] if row['结算电量/容量'] != 0 else 0
            price = round(price/1000, 4)
            # 将数据添加到新的DataFrame中
            new_row = pd.DataFrame({'结算科目': [row['结算科目']],
                                    '结算科目编码': [code],
                                    '电量(万kWh)': round([row['结算电量/容量']][0]/10, 2),
                                    r'电价(元/kWh)': [price],
                                    '结算电费(万元)': round([row['结算电费']][0]/10000, 2)})
            if len(df_summary) == 0:
                df_summary = new_row
            else:
                df_summary = pd.concat([df_summary, new_row], ignore_index=True)

    df_summary['一级索引'] = ''
    df_summary['一级索引'] = df_summary['一级索引'].astype(str)
    df_summary.loc[df_summary['结算科目编码'].str.len() == 8, '一级索引'] = '中长期交易'
    df_summary['一级索引'] = df_summary['一级索引'].replace('', np.nan)
    df_summary['一级索引'].fillna(df_summary['结算科目'], inplace=True)
    df_summary.set_index(['一级索引', '结算科目'], inplace=True)

    df_summary['电量占比%'] = np.nan
    quantity_tmp = df_summary.loc['中长期交易', '电量(万kWh)'].sum()
    df_summary.loc['中长期交易', '电量占比%'] = (df_summary.loc['中长期交易', '电量(万kWh)']/quantity_tmp).values
    df_summary['电量占比%'] = df_summary['电量占比%'].apply(lambda x: format(x, '.2%'))

    # 计算结算电量，结算电量的计算方法，将结算科目编码为01开头行的电量相加
    idx = df_summary['结算科目编码'].str.startswith('01')
    quantity_tmp = df_summary.loc[idx, '电量(万kWh)'].sum()
    df_summary['结算电量(万kWh)'] = df_summary['电量(万kWh)']
    df_summary.loc[idx, '结算电量(万kWh)'] = quantity_tmp

    # 计算结算金额，结算电费的计算方法，将结算科目编码为01开头行的电费相加
    fee_tmp = df_summary.loc[idx, '结算电费(万元)'].sum()
    df_summary['结算金额(万元)'] = df_summary['结算电费(万元)']
    df_summary.loc[idx, '结算金额(万元)'] = fee_tmp

    # 计算合计金额
    df_summary['合计金额(万元)'] = round(df_summary['结算电费(万元)'].sum(), 2)

    # 计算综合结算均价
    df_summary['综合结算均价(元/kWh)'] = round(df_summary['合计金额(万元)'].iloc[0] /
                                               df_summary['结算电量(万kWh)'].iloc[0], 4)

    # 计算电能交易均价
    df_summary['电能交易均价(元/kWh)'] = round(df_summary['结算金额(万元)'].iloc[0] /
                                               df_summary['结算电量(万kWh)'].iloc[0], 4)

    # 删掉"结算科目编码"列
    df_summary.drop(columns='结算科目编码', inplace=True)
    df_summary.index.names = ['', '']
    df_summary = df_summary.replace('nan%', '0%')
    
    return df_summary

In [8]:
df_summary = summary_settlement(df_for_plot)
df_summary.to_clipboard()